In [ ]:
!python -m nltk.downloader punkt
!pip install stanza

In [ ]:
import pandas as pd
import re
import math
import string
from collections import Counter
import stanza

stanza.download("ru") 
nlp = stanza.Pipeline("ru", processors="tokenize,pos,lemma")

In [ ]:
df = pd.read_excel('корпус.xlsx')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Очистка текстов от символов генерации и лишних пробелов
def clean_text(text):
    text = str(text)
    text = re.sub(r"[*#_`]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["Текст"] = df["Текст"].apply(clean_text)

In [ ]:
docs = [nlp(text) for text in df["Текст"]]

In [ ]:
# Количество слов
def word_count(doc):
    return sum(1 for sent in doc.sentences for w in sent.words if w.upos != "PUNCT")

df["word_count"] = [word_count(d) for d in docs]

In [ ]:
# Средняя длина слова
def avg_word_length(doc):
    words = [w.text for s in doc.sentences for w in s.words if w.text.isalpha()]
    return sum(len(w) for w in words) / len(words) if words else 0

df["avg_word_length"] = [avg_word_length(d) for d in docs]

In [ ]:
# Средняя длина предложения
def avg_sentence_length(doc):
    
    lengths = []
    
    for s in doc.sentences:
        words = [w.text for w in s.words if w.text.isalpha()]
        if words:
            lengths.append(len(words))
    
    return sum(lengths) / len(lengths) if lengths else 0

df["avg_sentence_length"] = [avg_sentence_length(d) for d in docs]

In [ ]:
def get_words(doc):
    return [w.lemma.lower() for s in doc.sentences for w in s.words if w.upos != "PUNCT" and w.lemma]

In [ ]:
# TTR
def ttr_doc(doc):
    words = get_words(doc)
    return len(set(words)) / len(words) if words else 0

df["TTR"] = [ttr_doc(d) for d in docs]

In [ ]:
# MSTTR
def msttr_doc(doc, segment_size=50):
    words = get_words(doc)
    
    if not words:
        return 0
    
    segments = [
        words[i:i + segment_size]
        for i in range(0, len(words), segment_size)
    ]
    
    ttr_vals = [
        len(set(seg)) / len(seg)
        for seg in segments if seg
    ]
    
    return sum(ttr_vals) / len(ttr_vals) if ttr_vals else 0


df["MSTTR"] = [msttr_doc(d) for d in docs]

In [ ]:
def hapax_ratio_lemmas(doc):

    lemmas = []
    
    for sent in doc.sentences:
        for word in sent.words:
            lemma = word.lemma
            
            if lemma and lemma.isalpha():
                lemmas.append(lemma.lower())
    
    if len(lemmas) == 0:
        return 0

    counts = Counter(lemmas)
    
    hapax = [l for l, c in counts.items() if c == 1]
    
    return len(hapax) / len(lemmas)

df["hapax_ratio"] = [hapax_ratio_lemmas(d) for d in docs]

In [ ]:
# Индекс Юла
def yule(doc):
    words = get_words(doc)
    
    if not words:
        return 0
    
    counts = Counter(words)
    N = len(words)
    
    frec = sum(f**2 for f in counts.values())
    
    return 10000 * (frec - N) / (N**2)

df["yule"] = [yule(d) for d in docs]

In [ ]:
def function_words(doc):
    words = 0
    fw = 0
    
    for s in doc.sentences:
        for w in s.words:
            if w.upos != "PUNCT":
                words += 1
                if w.upos in {"ADP","CCONJ","SCONJ","PART","INTJ"}:
                    fw += 1
    
    return fw / words if words else 0

df["function_words"] = [function_words(d) for d in docs]

In [ ]:
# Herdan's C (LogTTR)
def herdan_c(text):
    words = [w for w in str(text).lower().split() if w.isalpha()]
    
    N = len(words)
    if N == 0:
        return 0

    V = len(set(words))
    
    if N == 0 or V == 0:
        return 0
    
    return math.log(V) / math.log(N)


df["herdan_c"] = df["Текст"].apply(herdan_c)

In [ ]:
# Энтропия Шеннона
def shannon_entropy_doc(doc):
    words = get_words(doc)
    
    if not words:
        return 0
    
    counts = Counter(words)
    N = len(words)
    
    return -sum(
        (f/N) * math.log2(f/N)
        for f in counts.values()
    )

df["shannon_entropy"] = [shannon_entropy_doc(d) for d in docs]

In [ ]:
# Индекс Симпсона
def simpson_index(text):
    words = [w for w in str(text).lower().split() if w.isalpha()]
    
    N = len(words)
    if N == 0:
        return 0

    counts = Counter(words)
    
    D = sum((freq / N) ** 2 for freq in counts.values())
    
    return D


df["simpson_index"] = df["Текст"].apply(simpson_index)

In [ ]:
# Распределение частей речи
def pos_distribution(doc):
    counts = Counter()
    total = 0
    
    for s in doc.sentences:
        for w in s.words:
            if w.upos:
                counts[w.upos] += 1
                total += 1
    
    return {k: v/total for k,v in counts.items()} if total else {}

pos_df = pd.DataFrame([pos_distribution(d) for d in docs]).fillna(0)
df = pd.concat([df, pos_df], axis=1)

In [ ]:
# Доля знаков препинания
punct = set(string.punctuation + "—«»…")

def punctuation_ratio(text, word_count):
    text = str(text)
    
    if word_count == 0:
        return 0
    
    punct_count = sum(1 for ch in text if ch in punct)
    
    return punct_count / word_count

df["punctuation_ratio"] = [punctuation_ratio(t, wc) for t, wc in zip(df["Текст"], df["word_count"])]

In [ ]:
# Доля цифр
def digit_ratio(text, word_count):
    text = str(text)
    
    if word_count == 0:
        return 0
    
    digit_count = sum(1 for ch in text if ch.isdigit())
    
    return digit_count / word_count

df["digit_ratio"] = [digit_ratio(t, wc) for t, wc in zip(df["Текст"], df["word_count"])]

In [ ]:
# Flesch
def count_syllables(word):
    vowels = "аеёиоуыэюя"
    return sum(1 for ch in word.lower() if ch in vowels)

def avg_syllables(text):
    words = re.findall(r"[а-яё]+", str(text).lower())
    
    if not words:
        return 0
    
    return sum(count_syllables(w) for w in words) / len(words)

def flesch_russian(text, avg_sent_len):
    ASW = avg_syllables(text)
    ASL = avg_sent_len
    
    if ASL == 0 or ASW == 0:
        return 0
    
    return 206.835 - 1.3 * ASL - 60.1 * ASW

df["flesch_ru"] = [flesch_russian(text, asl) for text, asl in zip(df["Текст"], df["avg_sentence_length"])]

In [ ]:
# Fog
def complex_word_ratio(text):
    words = re.findall(r"[а-яё]+", str(text).lower())
    
    if not words:
        return 0
    
    complex_words = [w for w in words if count_syllables(w) >= 3]
    
    return len(complex_words) / len(words)

def gunning_fog_russian(text, avg_sent_len):
    ASL = avg_sent_len
    PCW = complex_word_ratio(text)
    
    if ASL == 0:
        return 0
    
    return 0.4 * (ASL + 100 * PCW)

df["gunning_fog_ru"] = [gunning_fog_russian(text, asl) for text, asl in zip(df["Текст"], df["avg_sentence_length"])]